# TinyLlama 1.1B — Portfolio Agent Fine-tune

LoRA fine-tune TinyLlama on pesnik.dev portfolio data.
Runs on Colab free T4 (16GB VRAM).

## 1. Install dependencies

In [ ]:
!pip install -qU torch transformers datasets accelerate peft trl bitsandbytes huggingface_hub

## 2. Clone portfolio repo + build dataset

In [ ]:
import os
os.environ["GIT_TERMINAL_PROMPT"] = "0"

# Clone (public repo — no auth needed)
!git clone --depth 1 https://github.com/pesnik/pesnik.dev.git
%cd pesnik.dev

# Build dataset
!python scripts/prepare_dataset.py --output data/train.jsonl
!head -2 data/train.jsonl | python -m json.tool --no-ensure-ascii 2>/dev/null || echo "jsonl ok"

## 3. Login to HuggingFace Hub (optional)

Run this cell only if you want to push the trained model to the Hub later.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Run training

Takes ~20-30 minutes on a T4. Uses QLoRA (4-bit).

In [ ]:
import torch

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "WARNING: no GPU")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f}GB" if torch.cuda.is_available() else "")

# Train
!python scripts/train_tinyllama.py \
  --dataset data/train.jsonl \
  --output-dir pesnik-tinyllama-lora \
  --num-epochs 3 \
  --batch-size 4 \
  --lr 2e-4

## 5. Push to HuggingFace Hub (optional)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "pesnik/portfolio-agent"
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(
    folder_path="pesnik-tinyllama-lora",
    repo_id=repo_id,
    repo_type="model",
)
print(f"✓ pushed to https://huggingface.co/{repo_id}")

## 6. Test inference

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base, "pesnik-tinyllama-lora")

def ask(question):
    prompt = f"<|user|>\n{question}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.7)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("<|assistant|>\n")[-1].strip()

tests = [
    "Who are you?",
    "What's your spirit animal?",
    "Tell me about Zygote.",
    "What are you building?",
]

for q in tests:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print()

## 7. Download model locally

In [ ]:
!zip -r pesnik-tinyllama-lora.zip pesnik-tinyllama-lora/
from google.colab import files
files.download("pesnik-tinyllama-lora.zip")